# Demo: spectral geometry

This demo focuses on the differentiable sparse eigensolvers of `cochain` and uses the eigensolver to investigate and manipulate the eigenvalues and eigenvectors of the Laplacian operators.

The examples below uses the LOBPCG eigensolver with a diagonally damped Cholesky preconditioner. To run these examples require the optional dependencies `pyvista`, `pytetwild`, `polyscope`, and `nvmath-python`, as well as CUDA-accelerated hardware.

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import polyscope as ps
import pyvista as pv
import torch
from jaxtyping import Float, Integer
from nvmath.sparse.advanced import DirectSolverOptions
from torch import Tensor

from cochain.complex import SimplicialMesh
from cochain.metric.tri import tri_hodge_stars, tri_masses
from cochain.sparse.decoupled_tensor import SparseDecoupledTensor, DiagDecoupledTensor
from cochain.sparse.linalg.eigen import LOBPCGConfig, LOBPCGPrecondConfig, lobpcg
from cochain.sparse.linalg.solvers import (
    DirectSolverConfig,
    NVMathDirectSolver,
)
from cochain.topology import betti
from cochain.vis import PolyscopeViewer

In [ ]:
# Update this variable to point to the cuDSS multithreading library,
# or set to None if not available.
MT_LIB_PATH = (
    os.environ["VIRTUAL_ENV"]
    + "/lib/python3.13/site-packages/nvidia/cu12/lib/libcudss_mtlayer_gomp.so.0"
)

nvmath_config = DirectSolverConfig(
    options=DirectSolverOptions(multithreading_lib=MT_LIB_PATH)
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
ps.init()

## Example 1: $L_0$ eigenvalue optimization on a tri mesh

For a given triangular mesh, we define the Hodge 0-Laplacian $L_0$ as

$$L_0 = \star_0^{-1} S_0 = \star_0^{-1} d_0^T M_1 d_0$$

where $\star_k$ is the diagonal Hodge $k$-star operator, $d_k$ is the $k$-th coboundary operator, $M_k$ is the consistent $k$-mass matrix, and $S_0$ is the stiffness matrix.

To investigate the spectrum of $L_0$, we define the generalized eigenvalue problem

$$S_0 v = \lambda \star_0 v$$

The multiplicity of the zero generalized eigenvalue of $S_0$ is related to the dimension of the 0-th homology group of the simplicial complex, while the nonzero eigenvalues and their associated eigenvectors provide a spatial frequency decomposition of the mesh geometry. In this example, we start by constructing a toroid triangular mesh, then push the lowest five nonzero eigenvalues of its $S_0$ into an equidistant series (i.e., $\lambda_i = i \lambda_1$ for $i$ = 1, 2, 3, 4, 5), and investigate the consequence of this spectral optimization on the geometry of the mesh.

Note that, for this example, $L_0$ is defined as a "hybrid" operator, where the 0-mass matrix $\star_0$ is defined using the discrete exterior calculus (DEC) framework, while the 1-mass matrix $M_1$ is defined using the finite element exterior calculus (FEEC) framework. In `cochain`, the tri mesh Laplacians defined in `metric.tri.tri_laplacians` follow the DEC definitions, but it is not too difficult to directly buid the hybrid $S_0$ here from the constituent operators.

In [ ]:
def compute_tri_hybrid_stiff_0(
    mesh: SimplicialMesh,
) -> Float[SparseDecoupledTensor, "vert vert"]:
    m1 = tri_masses.mass_1(mesh)
    d0 = mesh.cbd[0]

    s0 = d0.T @ m1 @ d0

    return s0

Lastly, we need a convenience function for computing the volume enclosed by a closed tri mesh. This function computes the enclosed volume by summing together the signed volumes of the tets formed by each tri and the origin. Assuming that the tris are oriented with a outward-facing normal (using the right-hand rule), then the "extra" volume cancels out to give the (positve) enclosed volume.

In [ ]:
def compute_enclosed_volume(
    tri_vert_coords: Float[Tensor, "tri local_vert=3 coord=3"],
) -> Float[Tensor, ""]:
    return torch.sum(
        tri_vert_coords[:, 0]
        * torch.linalg.cross(tri_vert_coords[:, 1], tri_vert_coords[:, 2], dim=-1)
    )

### Mesh generation

Generate the toroid tri mesh via `pyvista` and load into `cochain`.

In [ ]:
toroid_pv_tri = (
    pv.ParametricSuperToroid(n1=0.75, n2=0.5, u_res=50, v_res=50)
    .smooth_taubin(n_iter=50, pass_band=0.1)
    .clean()
    .triangulate()
)

In [ ]:
toroid = SimplicialMesh.from_tri_mesh(
    vert_coords=torch.from_numpy(np.asarray(toroid_pv_tri.points)).to(
        dtype=torch.float64
    ),
    tris=torch.from_numpy(np.asarray(toroid_pv_tri.regular_faces)).to(
        dtype=torch.int64
    ),
).to(device)

Visualize the mesh structure using `polyscope`.

In [ ]:
ps.remove_all_structures()

In [ ]:
init_toroid_view = PolyscopeViewer(name="init_toroid", mesh=toroid)

In [ ]:
ps.show()  # close the vis window to continue

Basic mesh quality sanity check:
* The mesh does not contain any boundaries.
* The mesh has the correct homology group dimensions for a 2D torus.
* The volume enclosed by the mesh is positive.

In [ ]:
assert not toroid.bd_vert_mask.any()
assert not toroid.bd_edge_mask.any()
assert not toroid.bd_tet_mask.any()

In [ ]:
b0, b1, b2 = betti.compute_betti_numbers(mesh=toroid, manifold=True)
assert b0 == 1
assert b1 == 2
assert b2 == 1

In [ ]:
init_vol = compute_enclosed_volume(toroid.vert_coords[toroid.tris])
assert init_vol > 0

### Regularization

In general, spectral geometry optimization problems are "underdetermined", in that there are typical multiple ways to deform the geometry of the mesh that will yield the same eigenvalue spectrum, some of which can lead to pathologies of mesh structure. Therefore, we employ three regularization techniques in this example to stabilize the optimization problem:

* Sobolev preconditioner as a low-pass filter of vertex coordinate gradient.
* Symmetric dirichlet energy penalty to improve mesh element (i.e., triangle) quality.
* Willmore energy penalty to prevent crumpling of the mesh surface.

#### Sobolev preconditioner

Let $L(v)$ be the loss function for the spectral optimization problem as a function of the input vertex coordinate matrix $v\in\mathbb R^{|V|\times 3}$. Then, calling `backward()` on the loss function computes the gradient $\bar v_{L^2} = dL/dv$ in the standard Euclidean/$L^2$ inner product space (i.e., the space of deformation vector fields). In general, there is no guarantee that the gradient $\bar v_{L^2}$ will deform the mesh smoothly; therefore, to prevent the gradient optimization from crumpling the mesh, it is better to regularize the gradient by mapping it to the Sobolev space $H^1$. In the Sobolev space, both the deformation vector field and its differential are bounded in the $L^2$ sense, and this puts constraints on the spatial variability of the deformation vector within each vertex neighborhood. Achieving this mapping requires solving the linear system

$$(M_0 + \tau S_0) \bar v_{H^1} = \bar v_{L^2}$$

To see why this leads to a smoother gradient, consider the eigendecomposition of $\bar v_{L^2} = \sum_i c_i M_0 \phi_i$. Then, for any eigenvector $\phi_i$, the action of $M_0 + \tau S_0$ on $\phi_i$ is

$$(M_0 + \tau S_0)\phi_i = (1 + \tau\lambda_i)M_0\phi_i$$

Therefore,

$$\bar v_{H^1} = (M_0 + \tau S_0)^{-1} \bar v_{L^2} = \sum_i \frac{c_i}{1 + \tau\lambda_i}\phi_i$$

Here, larger eigenvalues correspond to higher frequency spatial modes, and their eigenvectors are downweighted by the inverse of their eigenvalues; $\tau$ serves to tune the strength of the regularization. Since the eigenvalues of $L_0$ scales inversely with the area of the mesh, a good strategy for tuning $\tau$ is to set $\tau\propto A$.

In [ ]:
def get_tri_sobolev_preconditioner(
    mesh: SimplicialMesh, tau: Float[Tensor, ""]
) -> NVMathDirectSolver:
    m0 = tri_hodge_stars.star_0(mesh).to_sdt()
    s0 = compute_tri_hybrid_stiff_0(mesh)

    op = SparseDecoupledTensor.assemble(tau * s0, m0)
    solver = NVMathDirectSolver(
        a=op, b=toroid.vert_coords, sparse_system_type="spd", config=nvmath_config
    )

    return solver

In [ ]:
init_tri_areas = tri_hodge_stars.compute_tri_areas(toroid.vert_coords, toroid.tris)
init_mesh_area = init_tri_areas.sum()

sobolev = get_tri_sobolev_preconditioner(toroid, init_mesh_area)

#### Symmetric Dirichlet energy

Consider a reference triangle and a deformed triangle, both defined on a 2D plane. The deformation gradient $F$ is defined to be the transformation that maps the reference triangle edges to the corresponding deformed triangle edges; i.e.,

$$F E_R = E_D$$

where $E_R$ and $E_D$ are the edge matrices for the reference and deformed triangles, respectively; here, we define the edge matrices to be $[e_{01}\, e_{02}]$ using the displacement vectors from the first vertex.

We can define the Dirichlet energy (density) associated with the transformation $F$ as

$$
\rho_D = \|F\|_F^2 = \sigma_1^2 + \sigma_2^2
$$

where $\sigma_1$ and $\sigma_2$ are the two singular values of $F$ that represent the principle stretch factors. To penalize both extreme stretching and flattening that produce sliver or degenerate triangles, we define a symmetric Dirichlet energy (density) as

$$
\rho_{SD} = \|F\|_F^2 + \|F^{-1}\|_F^2 = \sigma_1^2 + \sigma_2^2 + 1/\sigma_1^2 + 1/\sigma_2^2
$$

Note that, in practice, it is more convenient to compute $\rho_{SD}$ using the trace definition of Frobenius norm,

$$
\rho_{SD} = \text{tr}[F^TF] + \text{tr}[(F^TF)^{-1}]
$$

In addition, note that the Dirichlet energy is naturally invariant to translation and rotation. The translational invariance comes from the fact that the edge matrix is defined by displacement vectors. The rotational invariance comes from the property of orthogonal matrices. By the polar decomposition, $F = RU$, where $R$ is a rotation matrix and $U$ is a scaling/shearing matrix; the $R$ cancels itself out in $\|F\|_F$ because, as an orthogonal matrix, $R^TR = I$.

Lastly, for a triangular mesh, the total symmetric Dirichlet energy is simply given by the surface integral of $\rho_{SD}$,

$$E_{SD} = \sum_i \rho_{SD,i}A_i$$

where $\rho_{SD,i}$ is the energy density on triangle $i$ relative to its reference triangle, and $A_i$ is the area of the reference triangle.

For this example, since we are working with triangle mesh elements embedded in $\mathbb R^3$, we define a local 2D coordinate frame for each triangle and use it to compute the edge matrix and deformation gradient. Specifically, for each triangle, we align the first vertex with the origin and the second vertex with the positive direction of the $x$-axis; then, the edge vectors $e_{01}'$ and $e_{02}'$ in this local 2D frame can be expressed using the original, 3D edge vectors as

$$
e_{01}' = \begin{bmatrix}
    \|e_{01}\| \\
    0
\end{bmatrix}
,\,
e_{02}' = \begin{bmatrix}
    (e_{01} \cdot e_{02})/\|e_{01}\| \\
    \|e_{01} \times e_{02}\|/\|e_{01}\|
\end{bmatrix}
$$

Note that this definition itself already makes the edge matrix $E$ rotationally invariant.

In [ ]:
def compute_tri_edge_matrix(
    vert_coords: Float[Tensor, "vert coord=3"],
    tris: Integer[Tensor, "tri local_vert=3"],
) -> Float[Tensor, "tri coord=2 local_edge=2"]:
    tri_verts = vert_coords[tris]

    e01 = tri_verts[:, 1] - tri_verts[:, 0]
    e02 = tri_verts[:, 2] - tri_verts[:, 0]

    e01_norm = torch.linalg.norm(e01, dim=-1)
    e02_proj_x_coord = torch.sum(e01 * e02, dim=-1) / e01_norm
    e02_proj_y_coord = (
        torch.linalg.norm(torch.linalg.cross(e01, e02, dim=-1), dim=-1) / e01_norm
    )

    e01_proj_coord = torch.stack([e01_norm, torch.zeros_like(e01_norm)], dim=-1)
    e02_proj_coord = torch.stack([e02_proj_x_coord, e02_proj_y_coord], dim=-1)

    edge_mat = torch.stack([e01_proj_coord, e02_proj_coord], dim=-1)

    return edge_mat


def compute_tri_deformation_gradient(
    edge_mat_inv_ref: Float[Tensor, "tri coord=2 local_edge=2"],
    vert_coords_deformed: Float[Tensor, "vert coord=3"],
    tris: Integer[Tensor, "tri local_vert=3"],
) -> Float[Tensor, "tri coord=2 coord=2"]:
    edge_mat_deformed = compute_tri_edge_matrix(vert_coords_deformed, tris)
    deform_grad = edge_mat_deformed @ edge_mat_inv_ref

    return deform_grad


def compute_tri_sym_dirichlet(
    edge_mat_inv_ref: Float[Tensor, "tri coord=2 local_edge=2"],
    ref_tri_areas: Float[Tensor, " tri"],
    vert_coords_deformed: Float[Tensor, "vert coord=3"],
    tris: Integer[Tensor, "tri local_vert=3"],
):
    deform_grad = compute_tri_deformation_gradient(
        edge_mat_inv_ref, vert_coords_deformed, tris
    )
    deform_grad_sym = torch.transpose(deform_grad, -1, -2) @ deform_grad
    deform_grad_sym_inv = torch.linalg.inv(deform_grad_sym)

    dirichlet = torch.einsum("bii->b", deform_grad_sym)
    dirichlet_inv = torch.einsum("bii->b", deform_grad_sym_inv)

    dirichlet_sym = torch.sum((dirichlet + dirichlet_inv) * ref_tri_areas)

    return dirichlet_sym

In this example, we define a single "idealized" reference triangle as an equilateral triangle with the area defined as the total tri mesh area divided by the number of tri elements, and precompute its edge matrix and the inv inverse edge matrix.

In [ ]:
ideal_tri_area = init_mesh_area / toroid.n_tris

In [ ]:
equilat_vert_coords = torch.tensor(
    [[0.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.5, math.sqrt(3.0) / 2.0, 0.0]],
    dtype=toroid.dtype,
    device=toroid.device,
) * math.sqrt(ideal_tri_area / (math.sqrt(3.0) / 4.0))

equilat_tri = torch.tensor([[0, 1, 2]], dtype=toroid.tris.dtype, device=toroid.device)

In [ ]:
edge_mat_ref = compute_tri_edge_matrix(equilat_vert_coords, equilat_tri)
edge_mat_inv_ref = torch.linalg.inv(edge_mat_ref)

#### Willmore energy

Although the symmetric Dirichlet energy preserves the integrity of individual triangle elements, it says nothing about the dihedral angles between adjacent triangles; therefore, to penalize mesh crumpling, we need an additional regularization term in the form of the Willmore energy, which measures the squared mean curvature over the mesh

$$W = \int_\Omega H^2\,dA$$

In the discrete setting, the mean curvature normal vector $n_H$ satisfies

$$S_0 v = 2 \star_0 n_H$$

or, equivalently, $n_H = \star_0^{-1} S_0 v /2$. Therefore, the discrete Willmore energy is

$$
W  = \sum_i \|n_{H,i}\|^2 [\star_0]_{ii}
= \text{tr}[n_H^T \star_0 n_H]
= \frac 1 4 \text{tr}[v^T S_0 \star_0^{-1} S_0 v]
$$

Here, $[\star_0]_{ii}$ is the diagonal element of $\star_0$ corresponding to the vertex area of the $i$-th vertex.

In [ ]:
def compute_tri_willmore(
    s0: Float[SparseDecoupledTensor, "vert vert"],
    m0: Float[DiagDecoupledTensor, "vert vert"],
    vert_coords: Float[Tensor, "vert coord=3"],
):
    return torch.trace(vert_coords.T @ s0 @ m0.inv @ s0 @ vert_coords)

### Visualization of the spectrum

Visualize the first five nonzero eigenvalues and their associated eigenvectors on the starting mesh.

In [ ]:
lobpcg_config = LOBPCGConfig(largest=False, maxiter=200, tol=1e-6)
precond_config = LOBPCGPrecondConfig(method="cholesky", nvmath_config=nvmath_config)

In [ ]:
m0_ddt = tri_hodge_stars.star_0(toroid)
m0 = m0_ddt.to_sdt()
s0 = compute_tri_hybrid_stiff_0(toroid)

In [ ]:
n_eigs = b0 + 5

In [ ]:
init_eig_vals, init_eig_vecs = lobpcg(
    a=s0,
    m=m0,
    n=2 * n_eigs,
    k=n_eigs,
    eps=0,
    lobpcg_config=lobpcg_config,
    precond_config=precond_config,
)

In [ ]:
init_eig_vals

In [ ]:
for i in range(1, n_eigs):
    init_toroid_view.add_k_cochain(
        k=0, name=f"eig_{i}", cochain=init_eig_vecs[:, i], datatype="symmetric"
    )

In [ ]:
ps.show()  # close the vis window to continue

### Spectral optimization

Here, we define the target eigenvalues such that $\lambda_i = i \lambda_1$ in terms of the first nonzero eigenvalue $\lambda_i$ of the original mesh, for $i$ = 1, 2, 3, 4, 5. Then, the loss function is defined as the log MSE between the current and target eigenvalues. Note that, instead of defining the loss in terms of the eigenvalues directly (which has a unit of $1/\text{length}^2$), we scale the eigenvalues by the mesh area to make the loss unitless. More importantly, scaling the vertex coordinates of a mesh by a factor of $s$ scales the mesh area by $s^2$ but the eigenvalues of $S_0$ by $s^{-2}$; therefore, defining the loss in terms of $\lambda_i A$ prevents the optimization from "cheating" by shrinking or inflating the mesh.

In [ ]:
fundamental = init_eig_vals[b0]
target_eig_vals = fundamental * torch.arange(
    b0, n_eigs, dtype=init_eig_vals.dtype, device=init_eig_vals.device
)

In [ ]:
def compute_tri_spectral_loss(
    eig_vals: Float[Tensor, " eig"],
    target_eig_vals: Float[Tensor, " eig"],
    mesh_area: Float[Tensor, ""],
    init_mesh_area: Float[Tensor, ""],
):
    return torch.sum(
        (torch.log(target_eig_vals * init_mesh_area) - torch.log(eig_vals * mesh_area))
        ** 2
    )

In [ ]:
n_steps = 200
update_frequency = 50

dirichlet_weight = 5e-3
willmore_weight = 1e-3

opt_lr = 10.0

eps = torch.finfo(toroid.vert_coords.dtype).eps

In [ ]:
toroid.requires_grad_()

optimizer = torch.optim.SGD([toroid.vert_coords], lr=opt_lr)

log_mse_traj = []
dirichlet_loss_traj = []
willmore_loss_traj = []

for t in range(n_steps):
    optimizer.zero_grad()

    # Compute the current eigenvalue spectrum.
    m0_ddt = tri_hodge_stars.star_0(toroid)
    m0 = m0_ddt.to_sdt()
    s0 = compute_tri_hybrid_stiff_0(toroid)

    eig_vals, eig_vecs = lobpcg(
        a=s0,
        m=m0,
        n=2 * n_eigs,
        k=n_eigs,
        eps=0,
        lobpcg_config=lobpcg_config,
        precond_config=precond_config,
    )

    # Compute the loss terms.
    mesh_area = tri_hodge_stars.compute_tri_areas(toroid.vert_coords, toroid.tris).sum()
    spectral_log_mse = compute_tri_spectral_loss(
        eig_vals[b0:], target_eig_vals, mesh_area, init_mesh_area
    )
    log_mse_traj.append(spectral_log_mse.detach().cpu().numpy())

    dirichlet_sym = dirichlet_weight * compute_tri_sym_dirichlet(
        edge_mat_inv_ref, ideal_tri_area, toroid.vert_coords, toroid.tris
    )
    dirichlet_loss_traj.append(dirichlet_sym.detach().cpu().numpy())

    willmore = willmore_weight * compute_tri_willmore(s0, m0_ddt, toroid.vert_coords)
    willmore_loss_traj.append(willmore.detach().cpu().numpy())

    loss = spectral_log_mse + dirichlet_sym + willmore
    loss.backward()

    # Compute the Sobolev gradient.
    with torch.no_grad():
        grad = toroid.grad.detach().clone()
        grad_smoothed = sobolev(grad, trans="N")
        toroid.grad = grad_smoothed.contiguous()

    optimizer.step()

    with torch.no_grad():
        # Recenter the mesh to avoid drift.
        toroid.vert_coords.sub_(toroid.vert_coords.mean(dim=0, keepdim=True))

        # Rescale the mesh to match the initial enclosed volume. This prevents
        # the optimizer from cheating by preserving the mesh area but deflating
        # it into a spike ball.
        updated_vol = compute_enclosed_volume(toroid.vert_coords[toroid.tris])
        vol_ratio = (init_vol / updated_vol).pow(1.0 / 3.0)
        toroid.vert_coords.mul_(vol_ratio)

        # Periodically update the Sobolev preconditioner to account for the
        # updated geometry.
        if t > 0 and t % update_frequency == 0:
            sobolev = get_tri_sobolev_preconditioner(toroid, mesh_area)

    # Log info.
    if t % update_frequency == 0:
        print(
            f"Timestep: {t:04d}; "
            f"spectral_log_mse: {spectral_log_mse:.4f}; "
            f"dirichlet: {dirichlet_sym:.4f}; "
            f"willmore: {willmore:.4f}"
        )

Visualize the optimization trajectory.

In [ ]:
log_mse_traj = np.asarray(log_mse_traj)
dirichlet_loss_traj = np.asarray(dirichlet_loss_traj)
willmore_loss_traj = np.asarray(willmore_loss_traj)

t_traj = np.asarray(range(len(log_mse_traj))) + 1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 4), layout="constrained")


axes[0].plot(t_traj, log_mse_traj, color="dodgerblue")
axes[0].set_xscale("log")
axes[0].set(xlabel="Time", ylabel="log_mse")


axes[1].set_xscale("log")
axes[1].plot(t_traj, willmore_loss_traj, color="dodgerblue")
axes[1].set(xlabel="Time", ylabel="willmore_loss")

axes[2].plot(t_traj, dirichlet_loss_traj, color="dodgerblue")
axes[2].set_xscale("log")
axes[2].set(xlabel="Time", ylabel="sym_dirichlet_loss")

plt.show()
plt.close()

Inspect the eigenvalue ratio after the optimization.

In [ ]:
with torch.no_grad():
    print(eig_vals[b0:] / eig_vals[b0])

Visualize the optimized mesh geometry and eigenvectors.

In [ ]:
opt_toroid_view = PolyscopeViewer(name="opt_mesh", mesh=toroid)

In [ ]:
for i in range(1, n_eigs):
    opt_toroid_view.add_k_cochain(
        k=0, name=f"eig_{i}", cochain=eig_vecs[:, i], datatype="symmetric"
    )

In [ ]:
ps.show()

Comparing the initial and optimized mesh geometry, it is notable that the optimizer is able to break the symmetry of the initial mesh (most notably represented by the fact that $\lambda_1$ initially has a multiplicity of 2); this results in a distorted toroid that is stretched out horizontally but compressed vertically.